# Side-Step — Google Colab Edition

**Fine-tune music LoRAs with [Side-Step](https://github.com/koda-dernet/Side-Step) directly in Google Colab.**

This notebook exposes the full Side-Step CLI pipeline -- preprocessing, training, AI captions, audio analysis, PP++ adaptive ranks, ComfyUI export -- with no local GPU required.

> **How to use:** Run cells top-to-bottom. Required sections are marked `[REQUIRED]`. Optional sections are marked `[OPTIONAL]`. Every `#@param` field has a hover tooltip.

---

| Section | | Description |
|---------|--|-------------|
| S0 GPU Check | `[REQUIRED]` | Verify GPU runtime |
| S1 Install & Mount | `[REQUIRED]` | Clone Side-Step, mount Drive |
| S2 Model Downloads | `[REQUIRED]` | Download ACE-Step checkpoints |
| S3 Settings | `[OPTIONAL]` | API keys for captions |
| S4 Dataset Prep | `[OPTIONAL]` | Audio scan, captions, tags |
| S5 Audio Analysis | `[OPTIONAL]` | BPM, key, time signature |
| S6 Preprocessing | `[REQUIRED]` | Audio to .pt tensors |
| S7 PP++ Analysis | `[OPTIONAL]` | Adaptive rank assignment |
| S8 Training | `[REQUIRED]` | Train adapter |
| S9 Export | `[OPTIONAL]` | ComfyUI .safetensors |
| S10 History | `[OPTIONAL]` | Past training runs |


---
## S0 — GPU Check `[REQUIRED]`
> If you see a red error below, go to **Runtime > Change runtime type > GPU** (T4 is fine, A100/L4 are faster).


In [ ]:
import torch
from IPython.display import HTML, display

_SS = {
    "bg":"#16161E","surface":"#1E1F28","panel":"#2D2E36","border":"#4A5F8F",
    "primary":"#5ccfe6","success":"#7ec97a","warning":"#ffcc66","error":"#f07178",
    "text":"#A1A8CF","bold":"#D5D8EC","muted":"#8995BC",
}

def _card(title, body, accent="primary"):
    c = _SS[accent]
    return HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {c};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{c};font-weight:bold;margin-bottom:6px;">{title}</div>'
        f'<div style="color:{_SS["text"]};">{body}</div></div>'
    )

def _table(rows):
    r = "".join(
        f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">{k}</td>'
        f'<td style="padding:3px 0;color:{_SS["bold"]};font-weight:600;">{v}</td></tr>'
        for k,v in rows
    )
    return f'<table style="border-collapse:collapse;">{r}</table>'

if not torch.cuda.is_available():
    display(_card("[FAIL] No GPU Detected",
        "Go to <b>Runtime > Change runtime type > GPU</b> and re-run this cell.", "error"))
    raise RuntimeError("No GPU runtime.")
else:
    _n = torch.cuda.get_device_name(0)
    _v = torch.cuda.get_device_properties(0).total_mem / 1024**3
    _c = torch.version.cuda or "N/A"
    display(_card("[OK] GPU Ready", _table([
        ("GPU", _n), ("VRAM", f"{_v:.1f} GB"), ("CUDA", _c), ("PyTorch", torch.__version__)
    ]), "success"))


---
## S1 — Install & Mount Drive `[REQUIRED]`
Clone the Side-Step repository, install dependencies, and mount Google Drive for persistent storage.

> Everything is saved to `My Drive/SideStep/` -- checkpoints, audio, tensors, adapters, and logs all survive runtime disconnects.


In [ ]:
#@title S1 -- Install Side-Step & Mount Google Drive { display-mode: "form" }
#@markdown Clones Side-Step from main, installs all dependencies, and creates
#@markdown a persistent directory tree on Google Drive.

import os, subprocess, sys
from pathlib import Path
from IPython.display import HTML, display

REPO_URL = "https://github.com/koda-dernet/Side-Step.git"
BRANCH = "main"  #@param {type:"string"} {tooltip: "Branch to clone. Use 'main' for stable releases."}
INSTALL_DIR = "/content/Side-Step"

if not Path(INSTALL_DIR).exists():
    print("[INFO] Cloning Side-Step (%s)..." % BRANCH)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, INSTALL_DIR],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print("[OK] Cloned.")
else:
    print("[OK] Side-Step already present at %s" % INSTALL_DIR)

print("[INFO] Installing dependencies (takes ~2 min on first run)...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{INSTALL_DIR}/requirements.txt"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
)
print("[OK] Dependencies installed.")

if INSTALL_DIR not in sys.path:
    sys.path.insert(0, INSTALL_DIR)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

SIDESTEP_ROOT = "/content/drive/MyDrive/SideStep"  #@param {type:"string"} {tooltip: "Root directory on Google Drive for all Side-Step files."}

_dirs = {
    "checkpoints": "ACE-Step model weights",
    "audio": "Raw audio files + .txt sidecars",
    "tensors": "Preprocessed .pt tensor files",
    "adapters": "Trained adapter weights",
    "logs": "TensorBoard run logs",
}
for name in _dirs:
    Path(f"{SIDESTEP_ROOT}/{name}").mkdir(parents=True, exist_ok=True)

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","success":"#7ec97a","text":"#A1A8CF","muted":"#8995BC","bold":"#D5D8EC"}
_rows = "".join(
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">{n}/</td>'
    f'<td style="padding:3px 0;color:{_SS["text"]};">{d}</td></tr>'
    for n, d in _dirs.items()
)
display(HTML(
    f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
    f'border-left:4px solid {_SS["primary"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
    f'<div style="color:{_SS["primary"]};font-weight:bold;margin-bottom:8px;">Drive Directory Tree</div>'
    f'<code style="color:{_SS["success"]};">{SIDESTEP_ROOT}/</code>'
    f'<table style="border-collapse:collapse;margin-top:6px;">{_rows}</table></div>'
))

from sidestep_engine.ui import set_plain_mode
set_plain_mode(True)
import torch
torch.set_float32_matmul_precision("medium")
print("\n[OK] Side-Step is ready.")


---
## S2 — Model Downloads `[REQUIRED]`
Download ACE-Step checkpoints to Google Drive. These persist across sessions -- you only download once.

| Model | HF Repo | Contents |
|-------|---------|----------|
| **ACE-Step 1.5 (shared)** | `ACE-Step/Ace-Step1.5` | Turbo variant, Qwen text embeddings, VAE |
| **ACE-Step base** | `ACE-Step/acestep-v15-base` | Base variant weights *(recommended for training)* |
| **ACE-Step SFT** | `ACE-Step/acestep-v15-sft` | SFT variant weights |
| **Qwen2.5-Omni-7B** | `Qwen/Qwen2.5-Omni-7B` | Local caption model (~15 GB, optional) |

> You **always** need the shared repo. Pick base (recommended) or sft. Qwen is only needed for local AI captions.


In [ ]:
#@title S2 -- Download Models { display-mode: "form" }
#@markdown Check the models you need. Already-downloaded models are skipped.

download_shared = True   #@param {type:"boolean"} {tooltip: "ACE-Step shared weights (turbo, VAE, Qwen embeddings). Always required."}
download_base   = True   #@param {type:"boolean"} {tooltip: "ACE-Step base variant. Recommended for training."}
download_sft    = False  #@param {type:"boolean"} {tooltip: "ACE-Step SFT variant. Better prompt adherence but lower LoRA quality."}
download_qwen   = False  #@param {type:"boolean"} {tooltip: "Qwen2.5-Omni-7B for local AI captions (~15 GB). Not needed if using Gemini/OpenAI."}

import os
from pathlib import Path
from IPython.display import HTML, display

CHECKPOINT_DIR = os.path.join(SIDESTEP_ROOT, "checkpoints")
_dl = []
if download_shared: _dl.append(("ACE-Step/Ace-Step1.5", "Ace-Step1.5"))
if download_base:   _dl.append(("ACE-Step/acestep-v15-base", "acestep-v15-base"))
if download_sft:    _dl.append(("ACE-Step/acestep-v15-sft", "acestep-v15-sft"))
if download_qwen:   _dl.append(("Qwen/Qwen2.5-Omni-7B", "Qwen2.5-Omni-7B"))

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","success":"#7ec97a","error":"#f07178","text":"#A1A8CF","bold":"#D5D8EC"}

if not _dl:
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["error"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["error"]};font-weight:bold;">[WARN] No models selected.</div>'
        f'<div style="color:{_SS["text"]};">You need at least the shared ACE-Step weights.</div></div>'
    ))
else:
    from huggingface_hub import snapshot_download
    for repo_id, folder_name in _dl:
        local_dir = os.path.join(CHECKPOINT_DIR, folder_name)
        if Path(local_dir).is_dir() and any(Path(local_dir).iterdir()):
            print(f"[OK] {repo_id} -- already present")
            continue
        print(f"[INFO] Downloading {repo_id}...")
        snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False)
        print(f"[OK] {repo_id} -- done")

    found = [d.name for d in Path(CHECKPOINT_DIR).iterdir() if d.is_dir()]
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Checkpoints Ready</div>'
        f'<div style="color:{_SS["text"]};">Found: <b>{", ".join(found)}</b></div></div>'
    ))


---
## S3 — Settings & API Keys `[OPTIONAL]`
Configure API keys for AI caption generation and lyrics fetching.

> You can use [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb) -- add secrets named `GEMINI_API_KEY`, `OPENAI_API_KEY`, or `GENIUS_API_TOKEN` and this cell picks them up.
>
> **Skip this** if you already have .txt sidecar files or are using pre-existing tensors.


In [ ]:
#@title S3 -- API Keys & Settings { display-mode: "form" }
#@markdown Configure caption and lyrics providers. Leave blank to skip.

#@markdown ---
#@markdown ### Caption Provider
gemini_api_key  = ""  #@param {type:"string"} {tooltip: "Google Gemini API key. Get one at https://aistudio.google.com/apikey"}
gemini_model    = "gemini-2.5-flash"  #@param ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-pro"] {tooltip: "Gemini model for caption generation."}
openai_api_key  = ""  #@param {type:"string"} {tooltip: "OpenAI API key. Alternative to Gemini."}
openai_model    = "gpt-4o"  #@param ["gpt-4o", "gpt-4o-mini", "gpt-4-turbo"] {tooltip: "OpenAI model for captions."}
openai_base_url = ""  #@param {type:"string"} {tooltip: "Custom OpenAI-compatible base URL. Leave blank for official OpenAI."}

#@markdown ---
#@markdown ### Lyrics & Metadata
genius_api_token   = ""  #@param {type:"string"} {tooltip: "Genius API token for lyrics. Get one at https://genius.com/api-clients"}
music_flamingo_url = ""  #@param {type:"string"} {tooltip: "Music Flamingo Gradio server URL for structured metadata. Leave blank to skip."}
hf_token           = ""  #@param {type:"string"} {tooltip: "HuggingFace token for authenticated Spaces."}

#@markdown ---
#@markdown ### Model Variant
model_variant = "base"  #@param ["base", "sft", "turbo"] {tooltip: "'base' recommended -- produces the best LoRAs."}

try:
    from google.colab import userdata
    if not gemini_api_key:   gemini_api_key   = userdata.get("GEMINI_API_KEY") or ""
    if not openai_api_key:   openai_api_key   = userdata.get("OPENAI_API_KEY") or ""
    if not genius_api_token: genius_api_token  = userdata.get("GENIUS_API_TOKEN") or ""
    if not hf_token:         hf_token          = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

import os
from pathlib import Path
from IPython.display import HTML, display

_variant_map = {"base":"acestep-v15-base","sft":"acestep-v15-sft","turbo":"Ace-Step1.5"}
CHECKPOINT_DIR = os.path.join(SIDESTEP_ROOT, "checkpoints")
_vp = Path(CHECKPOINT_DIR) / _variant_map.get(model_variant, model_variant)
if not _vp.is_dir():
    print(f"[WARN] Variant folder not found: {_vp}")
    print(f"       Available: {[d.name for d in Path(CHECKPOINT_DIR).iterdir() if d.is_dir()]}")
else:
    print(f"[OK] Model variant: {model_variant} -> {_vp}")

from sidestep_engine.settings import load_settings, save_settings, _default_settings
settings = load_settings() or _default_settings()
settings["checkpoint_dir"] = CHECKPOINT_DIR
settings["trained_adapters_dir"] = os.path.join(SIDESTEP_ROOT, "adapters")
settings["preprocessed_tensors_dir"] = os.path.join(SIDESTEP_ROOT, "tensors")
for k, v in [("gemini_api_key",gemini_api_key),("gemini_model",gemini_model),
             ("openai_api_key",openai_api_key),("openai_model",openai_model),
             ("openai_base_url",openai_base_url),("genius_api_token",genius_api_token),
             ("music_flamingo_url",music_flamingo_url),("hf_token",hf_token)]:
    if v: settings[k] = v
settings["first_run_complete"] = True
save_settings(settings)

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","success":"#7ec97a","text":"#A1A8CF","muted":"#8995BC","bold":"#D5D8EC"}
_ks = []
_ks.append(f'<span style="color:{_SS["success"] if gemini_api_key else _SS["muted"]};">Gemini {"[set]" if gemini_api_key else "[--]"}</span>')
_ks.append(f'<span style="color:{_SS["success"] if openai_api_key else _SS["muted"]};">OpenAI {"[set]" if openai_api_key else "[--]"}</span>')
_ks.append(f'<span style="color:{_SS["success"] if genius_api_token else _SS["muted"]};">Genius {"[set]" if genius_api_token else "[--]"}</span>')
_ks.append(f'<span style="color:{_SS["success"] if music_flamingo_url else _SS["muted"]};">Flamingo {"[set]" if music_flamingo_url else "[--]"}</span>')
display(HTML(
    f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
    f'border-left:4px solid {_SS["primary"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
    f'<div style="color:{_SS["primary"]};font-weight:bold;margin-bottom:6px;">Settings Saved</div>'
    f'<div style="color:{_SS["text"]};">API Keys: {" &nbsp;|&nbsp; ".join(_ks)}</div>'
    f'<div style="color:{_SS["text"]};margin-top:4px;">Model variant: <b style="color:{_SS["bold"]};">{model_variant}</b></div></div>'
))


---
## S4 — Dataset Preparation `[OPTIONAL]`
Upload audio files, generate AI captions, manage trigger tags, and build a dataset.

> Place `.wav`/`.mp3`/`.flac` files in `audio/` on Drive. Each can have a matching `.txt` sidecar with metadata. See the [documentation](https://github.com/koda-dernet/Side-Step/blob/main/sidestep_documentation/Dataset%%20Preparation.md) for sidecar format.
>
> **Skip this** if you already have preprocessed `.pt` tensors.


In [ ]:
#@title S4a -- Scan Audio Files { display-mode: "form" }
#@markdown Point to your audio folder and see what's there.

audio_dir = "/content/drive/MyDrive/SideStep/audio"  #@param {type:"string"} {tooltip: "Folder containing audio files (.wav, .mp3, .flac, .ogg, .m4a)."}

from pathlib import Path
from IPython.display import HTML, display

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","success":"#7ec97a","warning":"#ffcc66","error":"#f07178","text":"#A1A8CF","muted":"#8995BC","bold":"#D5D8EC","border_s":"#3a3f4f"}
_ae = {".wav",".mp3",".flac",".ogg",".m4a",".aac"}
_ap = Path(audio_dir)

if not _ap.is_dir():
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["error"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["error"]};font-weight:bold;">[FAIL] Not found: {audio_dir}</div></div>'
    ))
else:
    _files = sorted(f for f in _ap.rglob("*") if f.is_file() and f.suffix.lower() in _ae)
    if not _files:
        display(HTML(
            f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
            f'border-left:4px solid {_SS["warning"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
            f'<div style="color:{_SS["warning"]};font-weight:bold;">[WARN] No audio files found in {audio_dir}</div></div>'
        ))
    else:
        from sidestep_engine.data.audio_duration import get_audio_duration
        _rows = []
        for f in _files:
            try:
                dur = get_audio_duration(str(f)); m, s = divmod(dur, 60); ds = f"{int(m)}m {int(s):02d}s"
            except Exception: ds = "?"
            sc = f'<span style="color:{_SS["success"]};">[yes]</span>' if f.with_suffix(".txt").is_file() else f'<span style="color:{_SS["muted"]};">[no]</span>'
            nm = f.relative_to(_ap) if f.is_relative_to(_ap) else f.name
            _rows.append(
                f'<tr style="border-bottom:1px solid {_SS["border_s"]};">'
                f'<td style="padding:4px 12px 4px 0;color:{_SS["text"]};">{nm}</td>'
                f'<td style="padding:4px 12px;color:{_SS["muted"]};">{ds}</td>'
                f'<td style="padding:4px 12px;text-align:center;">{sc}</td></tr>'
            )
        display(HTML(
            f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
            f'border-left:4px solid {_SS["primary"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
            f'<div style="color:{_SS["primary"]};font-weight:bold;margin-bottom:8px;">{len(_files)} Audio File(s)</div>'
            f'<table style="border-collapse:collapse;width:100%;">'
            f'<tr style="border-bottom:1px solid {_SS["border"]};">'
            f'<th style="text-align:left;padding:4px 12px 4px 0;color:{_SS["muted"]};">File</th>'
            f'<th style="text-align:left;padding:4px 12px;color:{_SS["muted"]};">Duration</th>'
            f'<th style="text-align:center;padding:4px 12px;color:{_SS["muted"]};">Sidecar</th></tr>'
            f'{"".join(_rows)}</table></div>'
        ))


In [ ]:
#@title S4b -- Generate AI Captions { display-mode: "form" }
#@markdown Generate captions and fetch lyrics for your audio files.

caption_provider  = "gemini"        #@param ["gemini", "openai", "local_8-10gb", "local_16gb", "music_flamingo", "lyrics_only", "none"] {tooltip: "'gemini'/'openai' = cloud API (need keys from S3). 'local_*' = Qwen on GPU (S2). 'music_flamingo' = remote server."}
caption_policy    = "fill_missing"  #@param ["fill_missing", "overwrite_caption", "overwrite_all"] {tooltip: "'fill_missing' = only files without captions. 'overwrite_caption' = redo captions keep lyrics."}
fetch_lyrics      = True            #@param {type:"boolean"} {tooltip: "Fetch lyrics from Genius alongside captions."}
lyrics_provider   = "genius"        #@param ["genius", "transcriber_server", "music_flamingo", "none"] {tooltip: "Lyrics source."}
default_artist    = ""              #@param {type:"string"} {tooltip: "Default artist for Genius lookups when filename lacks 'Artist - Title'."}
use_google_search = False           #@param {type:"boolean"} {tooltip: "Grounding with Google Search for Gemini (adds per-query cost)."}

#@markdown ---
#@markdown Uses `audio_dir` from S4a.

from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import HTML, display
from sidestep_engine.data.enrich_song import enrich_one
from sidestep_engine.settings import (
    get_gemini_api_key, get_gemini_model, get_genius_api_token,
    get_openai_api_key, get_openai_model, get_openai_base_url,
    get_hf_token, get_music_flamingo_url,
)

_ae = {".wav",".mp3",".flac",".ogg",".m4a",".aac"}
_files = sorted(f for f in Path(audio_dir).rglob("*") if f.is_file() and f.suffix.lower() in _ae)
if not _files:
    print("[FAIL] No audio files. Run S4a first.")
else:
    _cap_fn = None
    if caption_provider == "gemini":
        _k = gemini_api_key or get_gemini_api_key(); _m = gemini_model or get_gemini_model() or "gemini-2.5-flash"
        if _k:
            from sidestep_engine.data.caption_provider_gemini import generate_caption as _gc
            _cap_fn = lambda t,a,l,p: _gc(t,a,_k,audio_path=p,lyrics_excerpt=l,model=_m,google_search=use_google_search)
        else: print("[FAIL] No Gemini key. Set in S3.")
    elif caption_provider == "openai":
        _k = openai_api_key or get_openai_api_key(); _m = openai_model or get_openai_model() or "gpt-4o"
        _b = openai_base_url or get_openai_base_url()
        if _k:
            from sidestep_engine.data.caption_provider_openai import generate_caption as _oc
            _cap_fn = lambda t,a,l,p: _oc(t,a,_k,audio_path=p,lyrics_excerpt=l,model=_m,base_url=_b)
        else: print("[FAIL] No OpenAI key. Set in S3.")
    elif caption_provider in ("local_8-10gb","local_16gb"):
        from sidestep_engine.data.caption_provider_local import generate_caption as _lc
        _tier = "8-10gb" if caption_provider == "local_8-10gb" else "16gb"
        _cap_fn = lambda t,a,l,p: _lc(t,a,audio_path=p,lyrics_excerpt=l,tier=_tier)

    _lyr_fn = None
    if fetch_lyrics and lyrics_provider == "genius":
        _t = genius_api_token or get_genius_api_token()
        if _t:
            from sidestep_engine.data.lyrics_provider_genius import fetch_lyrics as _gl
            _lyr_fn = lambda a,t: _gl(a,t,_t)
        else: print("[WARN] No Genius token -- lyrics disabled.")

    _meta_fn = None
    _fu = music_flamingo_url or get_music_flamingo_url() or ""
    if _fu:
        _ht = hf_token or get_hf_token() or ""
        from sidestep_engine.data.metadata_provider_music_flamingo import fetch_music_flamingo_metadata as _mfm
        _meta_fn = lambda p: _mfm(str(p),server_url=_fu,hf_token=_ht or None)

    w, s, fl = 0, 0, 0
    for af in tqdm(_files, desc="Captioning", unit="file"):
        r = enrich_one(af,default_artist=default_artist,caption_fn=_cap_fn,lyrics_fn=_lyr_fn,metadata_fn=_meta_fn,policy=caption_policy)
        st = r.get("status",""); w += st=="written"; s += st=="skipped"; fl += st not in ("written","skipped")

    if caption_provider in ("local_8-10gb","local_16gb"):
        try:
            from sidestep_engine.data.caption_provider_local import unload_model; unload_model()
        except Exception: pass

    _SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","text":"#A1A8CF"}
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Captions Complete</div>'
        f'<div style="color:{_SS["text"]};">Written: <b>{w}</b> | Skipped: <b>{s}</b> | Failed: <b>{fl}</b></div></div>'
    ))


In [ ]:
#@title S4c -- Manage Sidecar Tags { display-mode: "form" }
#@markdown Add, remove, or list trigger tags in .txt sidecar files.

tag_action   = "list"    #@param ["list", "add", "remove", "clear"] {tooltip: "'list' = show tags. 'add'/'remove' = modify. 'clear' = remove all."}
tag_text     = ""        #@param {type:"string"} {tooltip: "The trigger tag (e.g. 'my_style')."}
tag_position = "prepend" #@param ["prepend", "append", "replace"] {tooltip: "'prepend' = first (recommended for triggers)."}

#@markdown ---
#@markdown Uses `audio_dir` from S4a.

from pathlib import Path
from sidestep_engine.data.sidecar_io import read_sidecar, write_sidecar, sidecar_path_for

_ae = {".wav",".mp3",".flac",".ogg",".m4a",".aac"}
_fs = sorted(f for f in Path(audio_dir).rglob("*") if f.is_file() and f.suffix.lower() in _ae)
if not _fs: print("[FAIL] No audio files.")
elif tag_action == "list":
    for af in _fs:
        d = read_sidecar(sidecar_path_for(af))
        t = (d.get("custom_tag") or d.get("trigger") or "").strip()
        print(f"  {af.name}: {t or '(none)'}")
elif tag_action == "add" and tag_text:
    c = 0
    for af in _fs:
        sc = sidecar_path_for(af); d = read_sidecar(sc)
        e = (d.get("custom_tag") or d.get("trigger") or "").strip()
        d["custom_tag"] = {"prepend":f"{tag_text} {e}","append":f"{e} {tag_text}","replace":tag_text}[tag_position].strip()
        d.pop("trigger",None); write_sidecar(sc,d); c += 1
    print(f"[OK] Added '{tag_text}' ({tag_position}) to {c} files")
elif tag_action == "remove" and tag_text:
    c = 0
    for af in _fs:
        sc = sidecar_path_for(af); d = read_sidecar(sc)
        e = (d.get("custom_tag") or d.get("trigger") or "").strip()
        if tag_text in e:
            d["custom_tag"] = " ".join(e.replace(tag_text,"").split())
            d.pop("trigger",None); write_sidecar(sc,d); c += 1
    print(f"[OK] Removed '{tag_text}' from {c} files")
elif tag_action == "clear":
    c = 0
    for af in _fs:
        sc = sidecar_path_for(af); d = read_sidecar(sc)
        if d.get("custom_tag") or d.get("trigger"):
            d["custom_tag"] = ""; d.pop("trigger",None); write_sidecar(sc,d); c += 1
    print(f"[OK] Cleared tags from {c} files")


In [ ]:
#@title S4d -- Build dataset.json { display-mode: "form" }
#@markdown Optional -- Side-Step reads .txt sidecars directly during preprocessing.

build_dataset_json = False        #@param {type:"boolean"} {tooltip: "Build a dataset.json. Leave False to skip."}
dataset_name       = "my_dataset" #@param {type:"string"} {tooltip: "Name for the dataset entry."}
trigger_tag        = ""           #@param {type:"string"} {tooltip: "Optional trigger tag to prepend to all captions."}
genre_ratio        = 0            #@param {type:"integer"} {tooltip: "Genre token percentage (0-100). 0 = none."}

#@markdown ---
#@markdown Uses `audio_dir` from S4a.

if build_dataset_json:
    from sidestep_engine.data.dataset_builder import build_dataset
    out_path, stats = build_dataset(input_dir=audio_dir, tag=trigger_tag, tag_position="prepend", name=dataset_name, output=None, genre_ratio=genre_ratio)
    print(f"[OK] Dataset: {stats['total']} samples, {stats['with_metadata']} with metadata -> {out_path}")
else:
    print("[INFO] Skipped. Side-Step reads .txt sidecars directly.")


---
## S5 — Audio Analysis `[OPTIONAL]`
Extract BPM, musical key, and time signature from audio files using local offline analysis.

> Results enrich .txt sidecars. Three modes:
> - **faf** -- fast (librosa only)
> - **mid** -- balanced (Demucs + librosa, recommended)
> - **sas** -- thorough (Demucs + deep multi-technique)


In [ ]:
#@title S5 -- Audio Analysis { display-mode: "form" }
#@markdown Extract BPM, key, and time signature.

run_audio_analysis = False          #@param {type:"boolean"} {tooltip: "Set True to run. Results written to .txt sidecars."}
analysis_mode      = "mid"          #@param ["faf", "mid", "sas"] {tooltip: "'faf' = fast. 'mid' = balanced (recommended). 'sas' = thorough."}
analysis_policy    = "fill_missing" #@param ["fill_missing", "overwrite_all"] {tooltip: "'fill_missing' = skip files with existing data."}
analysis_chunks    = 5              #@param {type:"integer"} {tooltip: "Chunks for 'sas' mode. More = better but slower."}

#@markdown ---
#@markdown Uses `audio_dir` from S4a.

if run_audio_analysis:
    from pathlib import Path
    from tqdm.notebook import tqdm
    from sidestep_engine.analysis.audio_analysis import analyze_audio
    from sidestep_engine.data.sidecar_io import merge_fields, read_sidecar, sidecar_path_for, write_sidecar
    from sidestep_engine.models.gpu_utils import clear_device_cache

    _ae = {".wav",".mp3",".flac",".ogg",".m4a",".aac"}
    _fs = sorted(f for f in Path(audio_dir).rglob("*") if f.is_file() and f.suffix.lower() in _ae)
    w, s, fl = 0, 0, 0
    for af in tqdm(_fs, desc="Analyzing", unit="file"):
        try:
            r = analyze_audio(af, device="cuda:0", mode=analysis_mode, n_chunks=analysis_chunks)
            r.pop("confidence", None)
            if not r: s += 1; continue
            sc = sidecar_path_for(af); ex = read_sidecar(sc)
            if analysis_policy == "fill_missing" and all(ex.get(k,"").strip() for k in ("bpm","key","signature")):
                s += 1; continue
            write_sidecar(sc, merge_fields(ex, r, policy=analysis_policy)); w += 1
        except Exception as e: fl += 1; print(f"  [FAIL] {af.name}: {e}")
        finally: clear_device_cache("cuda:0")
    print(f"\n[OK] Analysis: {w} written, {s} skipped, {fl} failed")
else:
    print("[INFO] Skipped.")


---
## S6 — Preprocessing `[REQUIRED]`
Convert raw audio into `.pt` tensor files (two-pass pipeline: VAE encoder, then DIT encoder).

> Models load sequentially to minimize VRAM. Output tensors on Drive can be reused across runs.
>
> **Already have `.pt` tensors?** Skip this and point S8 directly to your tensor folder.


In [ ]:
#@title S6 -- Preprocess Audio to Tensors { display-mode: "form" }
#@markdown Two-pass pipeline: VAE encodes waveforms, then DIT encoder produces training-ready tensors.

#@markdown ### Paths
preprocess_audio_dir    = "/content/drive/MyDrive/SideStep/audio"              #@param {type:"string"} {tooltip: "Directory with raw audio files to preprocess."}
preprocess_output_dir   = "/content/drive/MyDrive/SideStep/tensors/my_dataset" #@param {type:"string"} {tooltip: "Where to save .pt tensors. Subfolder per dataset recommended."}
preprocess_dataset_json = ""  #@param {type:"string"} {tooltip: "Optional dataset.json path. Leave blank to read sidecars directly."}

#@markdown ### Options
normalize_mode = "peak"  #@param ["none", "peak", "lufs"] {tooltip: "'peak' = -1.0 dBFS (recommended). 'lufs' = -14 LUFS. 'none' = skip."}
max_duration   = 0       #@param {type:"integer"} {tooltip: "Max audio duration in seconds. 0 = auto-detect."}

#@markdown ---
#@markdown Checkpoint and model variant from S2/S3.

import os
from IPython.display import HTML, display
from sidestep_engine.data.preprocess import preprocess_audio_files
from sidestep_engine.models.gpu_utils import clear_device_cache

print("=" * 60)
print("  Preprocessing")
print("=" * 60)
print(f"  Source:     {preprocess_audio_dir}")
if preprocess_dataset_json: print(f"  JSON:       {preprocess_dataset_json}")
print(f"  Output:     {preprocess_output_dir}")
print(f"  Checkpoint: {CHECKPOINT_DIR}")
print(f"  Variant:    {model_variant}")
print(f"  Normalize:  {normalize_mode}")
print(f"  Max dur:    {'auto' if max_duration <= 0 else f'{max_duration}s'}")
print("=" * 60)

_preprocessed_tensor_dir = preprocess_output_dir
try:
    result = preprocess_audio_files(
        audio_dir=preprocess_audio_dir, output_dir=preprocess_output_dir,
        checkpoint_dir=CHECKPOINT_DIR, variant=model_variant,
        max_duration=max_duration, dataset_json=preprocess_dataset_json or None,
        device="cuda:0", precision="auto", normalize=normalize_mode,
        target_db=-1.0, target_lufs=-14.0,
    )
    _preprocessed_tensor_dir = result["output_dir"]
    _SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","error":"#f07178","text":"#A1A8CF","bold":"#D5D8EC"}
    _fail = f'<div style="color:{_SS["error"]};margin-top:4px;">Failed: <b>{result["failed"]}</b></div>' if result.get("failed") else ""
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Preprocessing Complete</div>'
        f'<div style="color:{_SS["text"]};">Processed: <b>{result["processed"]}/{result["total"]}</b></div>'
        f'{_fail}'
        f'<div style="color:{_SS["text"]};margin-top:4px;">Output: <code>{result["output_dir"]}</code></div></div>'
    ))
except Exception as exc:
    print(f"\n[FAIL] Preprocessing failed: {exc}")
    import traceback; traceback.print_exc()
finally:
    clear_device_cache("cuda:0")


---
## S7 — PP++ / Fisher Analysis `[OPTIONAL]`
Compute per-module importance and assign adaptive LoRA ranks.

> PP++ uses Fisher Information + Spectral Analysis to assign higher ranks to important layers and lower ranks to unimportant ones. Only compatible with LoRA and DoRA.


In [ ]:
#@title S7 -- PP++ Fisher Analysis { display-mode: "form" }
#@markdown Adaptive rank assignment based on your dataset.

run_pp_analysis   = False      #@param {type:"boolean"} {tooltip: "Run PP++ analysis. Saves fisher_map.json alongside tensors."}
pp_base_rank      = 64         #@param {type:"integer"} {tooltip: "Base rank budget. PP++ distributes ranks around this."}
pp_rank_min       = 16         #@param {type:"integer"} {tooltip: "Minimum rank for any module."}
pp_rank_max       = 128        #@param {type:"integer"} {tooltip: "Maximum rank for any module."}
pp_timestep_focus = "balanced" #@param ["balanced", "early", "mid", "late"] {tooltip: "'balanced' = all timesteps equally."}

#@markdown ---
#@markdown Dataset auto-filled from S6.

if run_pp_analysis:
    _ds = _preprocessed_tensor_dir if "_preprocessed_tensor_dir" in dir() else preprocess_output_dir
    from sidestep_engine.analysis.fisher import run_fisher_analysis
    from sidestep_engine.models.gpu_utils import clear_device_cache
    print(f"[INFO] Running PP++ on: {_ds}")
    print(f"       Rank budget: {pp_base_rank} (range {pp_rank_min}--{pp_rank_max})")
    try:
        r = run_fisher_analysis(checkpoint_dir=CHECKPOINT_DIR, variant=model_variant, dataset_dir=_ds,
            base_rank=pp_base_rank, rank_min=pp_rank_min, rank_max=pp_rank_max,
            timestep_focus=pp_timestep_focus, auto_confirm=True)
        if r:
            n = len(r.get("rank_pattern",{})); b = r.get("rank_budget",{})
            print(f"\n[OK] PP++: {n} modules, ranks {b.get('min','?')}--{b.get('max','?')}")
        else: print("[WARN] Analysis produced no results.")
    except Exception as e: print(f"[FAIL] {e}"); import traceback; traceback.print_exc()
    finally: clear_device_cache("cuda:0")
else:
    print("[INFO] Skipped.")


---
## S8 — Training `[REQUIRED]`
Train your adapter. This is the main event.

> Start with defaults, then tweak. After training, export for ComfyUI in S9.
>
> **Resume a run:** Set `resume_from` to a checkpoint path on Drive.


In [ ]:
#@title S8a -- Training Configuration { display-mode: "form" }
#@markdown Configure all training parameters. Hover for tooltip explanations.

#@markdown ---
#@markdown ### Required
dataset_dir = "/content/drive/MyDrive/SideStep/tensors/my_dataset"  #@param {type:"string"} {tooltip: "Path to .pt tensors (S6 output) or raw audio folder."}
output_dir  = "/content/drive/MyDrive/SideStep/adapters/my_lora"    #@param {type:"string"} {tooltip: "Where to save adapter weights and checkpoints."}
run_name    = ""  #@param {type:"string"} {tooltip: "Run name for TensorBoard and folder naming. Blank = auto."}

#@markdown ---
#@markdown ### Adapter
adapter_type   = "lora"  #@param ["lora", "dora", "lokr", "loha", "oft"] {tooltip: "'lora'/'dora' = PEFT (PP++ compatible). 'lokr'/'loha' = LyCORIS. 'oft' = experimental."}
rank           = 64      #@param {type:"integer"} {tooltip: "LoRA rank. Higher = more expressive. 32-128 typical."}
alpha          = 128     #@param {type:"integer"} {tooltip: "LoRA alpha. Convention: alpha = 2x rank."}
dropout        = 0.1     #@param {type:"number"} {tooltip: "Dropout. 0.05-0.15 prevents overfitting."}
attention_type = "both"  #@param ["both", "self", "cross"] {tooltip: "'both' = self + cross attention (recommended)."}
target_mlp     = True    #@param {type:"boolean"} {tooltip: "Also target MLP/FFN layers."}

#@markdown ---
#@markdown ### Training
learning_rate         = 1e-4  #@param {type:"number"} {tooltip: "LR. 1e-4 default. 5e-5 for PP++."}
batch_size            = 1     #@param {type:"integer"} {tooltip: "Samples per forward pass. 1 typical on T4."}
gradient_accumulation = 4     #@param {type:"integer"} {tooltip: "Accumulate N steps. Effective batch = batch x this."}
epochs                = 100   #@param {type:"integer"} {tooltip: "Full dataset passes. 50-200 typical."}
warmup_steps          = 100   #@param {type:"integer"} {tooltip: "LR ramp from 0 to target."}
max_steps             = 0     #@param {type:"integer"} {tooltip: "Hard stop after N steps. 0 = use epochs."}
seed                  = 42    #@param {type:"integer"} {tooltip: "Random seed."}

#@markdown ---
#@markdown ### Checkpointing
save_every      = 10    #@param {type:"integer"} {tooltip: "Checkpoint every N epochs."}
save_best       = True  #@param {type:"boolean"} {tooltip: "Keep best-loss checkpoint."}
log_every       = 10    #@param {type:"integer"} {tooltip: "TensorBoard log every N steps."}
log_heavy_every = 50    #@param {type:"integer"} {tooltip: "Heavy metrics every N steps. 0 = off."}
resume_from     = ""    #@param {type:"string"} {tooltip: "Checkpoint path to resume from. Blank = fresh."}

#@markdown ---
#@markdown ### Advanced
optimizer_type         = "adamw"   #@param ["adamw", "adamw8bit", "prodigy", "sgd"] {tooltip: "'adamw' standard. 'adamw8bit' saves VRAM."}
scheduler_type         = "cosine"  #@param ["cosine", "linear", "constant", "cosine_with_restarts"] {tooltip: "'cosine' recommended."}
gradient_checkpointing = True      #@param {type:"boolean"} {tooltip: "Trade compute for VRAM. Essential on T4."}
offload_encoder        = False     #@param {type:"boolean"} {tooltip: "Offload text encoder to CPU. Saves ~2GB VRAM."}
weight_decay           = 0.01      #@param {type:"number"} {tooltip: "L2 regularization."}
max_grad_norm          = 1.0       #@param {type:"number"} {tooltip: "Gradient clipping norm."}
loss_weighting         = "none"    #@param ["none", "snr", "min_snr"] {tooltip: "Loss weighting strategy."}
cfg_ratio              = 0.15      #@param {type:"number"} {tooltip: "CFG dropout ratio. 0.1-0.2 typical."}
dataset_repeats        = 1         #@param {type:"integer"} {tooltip: "Repeat dataset N times per epoch."}

#@markdown ---
#@markdown Checkpoint and variant from S2/S3.

import os, datetime
from pathlib import Path
from IPython.display import HTML, display

if "_preprocessed_tensor_dir" in dir() and _preprocessed_tensor_dir:
    dataset_dir = _preprocessed_tensor_dir

if not run_name:
    _base = Path(dataset_dir).name if dataset_dir else "run"
    run_name = f"{_base}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"

if output_dir.endswith("my_lora"):
    output_dir = os.path.join(SIDESTEP_ROOT, "adapters", adapter_type, run_name)

_log_dir = os.path.join(SIDESTEP_ROOT, "logs", run_name)
os.makedirs(_log_dir, exist_ok=True)

_params = dict(
    checkpoint_dir=CHECKPOINT_DIR, model_variant=model_variant, base_model=model_variant,
    dataset_dir=dataset_dir, output_dir=output_dir, adapter_type=adapter_type,
    rank=rank, alpha=alpha, dropout=dropout, attention_type=attention_type, target_mlp=target_mlp,
    learning_rate=learning_rate, batch_size=batch_size, gradient_accumulation=gradient_accumulation,
    epochs=epochs, warmup_steps=warmup_steps, max_steps=max_steps, seed=seed,
    save_every=save_every, save_best=save_best, log_every=log_every, log_heavy_every=log_heavy_every,
    log_dir=_log_dir, run_name=run_name, resume_from=resume_from or None,
    optimizer_type=optimizer_type, scheduler_type=scheduler_type,
    gradient_checkpointing=gradient_checkpointing, offload_encoder=offload_encoder,
    weight_decay=weight_decay, max_grad_norm=max_grad_norm, loss_weighting=loss_weighting,
    cfg_ratio=cfg_ratio, dataset_repeats=dataset_repeats, device="cuda:0", precision="auto",
)

from sidestep_engine.cli.common import build_configs_from_dict
adapter_cfg, train_cfg = build_configs_from_dict(_params)

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","text":"#A1A8CF","muted":"#8995BC","bold":"#D5D8EC"}
display(HTML(
    f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
    f'border-left:4px solid {_SS["primary"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
    f'<div style="color:{_SS["primary"]};font-weight:bold;margin-bottom:8px;">Training Config</div>'
    f'<table style="border-collapse:collapse;">'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Adapter</td><td style="color:{_SS["bold"]};"><b>{adapter_type}</b> (rank={rank}, alpha={alpha})</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Model</td><td style="color:{_SS["bold"]};"><b>{model_variant}</b></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Dataset</td><td style="color:{_SS["text"]};"><code>{dataset_dir}</code></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Output</td><td style="color:{_SS["text"]};"><code>{output_dir}</code></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Run</td><td style="color:{_SS["bold"]};"><b>{run_name}</b></td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">LR</td><td style="color:{_SS["text"]};">{learning_rate} ({optimizer_type} + {scheduler_type})</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Batch</td><td style="color:{_SS["text"]};">{batch_size} x {gradient_accumulation} accum</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Epochs</td><td style="color:{_SS["text"]};">{epochs}{f" (max {max_steps} steps)" if max_steps else ""}</td></tr>'
    f'<tr><td style="padding:3px 16px 3px 0;color:{_SS["muted"]};">Logs</td><td style="color:{_SS["text"]};"><code>{_log_dir}</code></td></tr>'
    f'</table></div>'
))
print("[OK] Config ready. Run next cell to train.")


In [ ]:
#@title S8b -- Start Training { display-mode: "form" }
#@markdown Press the stop button to interrupt gracefully.

from tqdm.notebook import tqdm
from IPython.display import HTML, display
from sidestep_engine.models.loader import load_decoder_for_training
from sidestep_engine.core.trainer import FixedLoRATrainer
from sidestep_engine.models.gpu_utils import clear_device_cache

model = None; trainer = None
try:
    print(f"[INFO] Loading model (variant={train_cfg.model_variant}, device={train_cfg.device})...")
    model = load_decoder_for_training(
        checkpoint_dir=train_cfg.checkpoint_dir, variant=train_cfg.model_variant,
        device=train_cfg.device, precision=train_cfg.precision,
    )
    print(f"[OK] Model loaded.")

    trainer = FixedLoRATrainer(model, adapter_cfg, train_cfg)
    _pbar = tqdm(total=train_cfg.max_epochs, desc="Training", unit="epoch")
    _cur_epoch = 0; _last_loss = 0.0

    for update in trainer.train():
        step, loss, msg = update.step, update.loss, update.msg
        if hasattr(update, "epoch") and update.epoch > _cur_epoch:
            _cur_epoch = update.epoch; _pbar.update(1)
        _last_loss = loss
        _pbar.set_postfix({"step": step, "loss": f"{loss:.6f}", "lr": f"{update.lr:.2e}" if hasattr(update,"lr") else "?"})
        if hasattr(update, "kind"):
            if update.kind == "checkpoint": print(f"\n[OK] Checkpoint saved: {update.checkpoint_path}")
            elif update.kind == "warn": print(f"\n[WARN] {msg}")
            elif update.kind == "complete": print(f"\n[OK] {msg}")
    _pbar.close()

    _SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","text":"#A1A8CF","bold":"#D5D8EC"}
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Training Complete</div>'
        f'<div style="color:{_SS["text"]};">Output: <code>{output_dir}</code></div>'
        f'<div style="color:{_SS["text"]};margin-top:4px;">Final loss: <b style="color:{_SS["bold"]};">{_last_loss:.6f}</b></div></div>'
    ))
    _trained_adapter_dir = output_dir

except KeyboardInterrupt:
    print("\n[INFO] Training interrupted. Partial checkpoints saved.")
    _trained_adapter_dir = output_dir
except Exception as exc:
    print(f"\n[FAIL] Training failed: {exc}")
    import traceback; traceback.print_exc()
    _trained_adapter_dir = output_dir
finally:
    del trainer; del model
    clear_device_cache("cuda:0")


In [ ]:
#@title S8c -- TensorBoard { display-mode: "form" }
#@markdown View training curves inline.

# The _log_dir variable is set by S8a.
%load_ext tensorboard
%tensorboard --logdir {_log_dir}


---
## S9 — Export to ComfyUI `[OPTIONAL]`
Convert your trained adapter to a single `.safetensors` file for ComfyUI.

> LyCORIS adapters (LoKR, LoHA) are already ComfyUI-compatible. PEFT adapters (LoRA, DoRA) need conversion.


In [ ]:
#@title S9 -- Export Adapter { display-mode: "form" }
#@markdown Export trained adapter for ComfyUI.

export_adapter_dir = ""        #@param {type:"string"} {tooltip: "Trained adapter directory. Auto-filled from S8."}
export_target      = "native"  #@param ["native", "generic"] {tooltip: "'native' = ComfyUI ACE-Step keys. 'generic' = diffusion_model prefix."}
export_normalize   = False     #@param {type:"boolean"} {tooltip: "Normalize alpha to match rank (strength=1.0 in ComfyUI)."}

from pathlib import Path
from IPython.display import HTML, display
from sidestep_engine.core.comfyui_export import export_for_comfyui

if not export_adapter_dir and "_trained_adapter_dir" in dir():
    export_adapter_dir = _trained_adapter_dir

_SS = {"panel":"#2D2E36","border":"#4A5F8F","success":"#7ec97a","error":"#f07178","text":"#A1A8CF"}

if not export_adapter_dir or not Path(export_adapter_dir).is_dir():
    print("[FAIL] No adapter directory. Train first (S8) or set path manually.")
else:
    result = export_for_comfyui(export_adapter_dir, target=export_target, normalize_alpha=export_normalize)
    if result["ok"]:
        display(HTML(
            f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
            f'border-left:4px solid {_SS["success"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
            f'<div style="color:{_SS["success"]};font-weight:bold;">[OK] Export Complete</div>'
            f'<div style="color:{_SS["text"]};">{result["message"]}</div></div>'
        ))
    else:
        print(f"[FAIL] {result['message']}")


---
## S10 — Run History `[OPTIONAL]`
View past training runs from your adapters directory.


In [ ]:
#@title S10 -- Run History { display-mode: "form" }
#@markdown List past training runs.

history_limit = 20  #@param {type:"integer"} {tooltip: "Maximum number of runs to show."}

from IPython.display import HTML, display
from sidestep_engine.gui.file_ops import build_history

runs = build_history()[:history_limit]

_SS = {"panel":"#2D2E36","border":"#4A5F8F","primary":"#5ccfe6","success":"#7ec97a",
       "text":"#A1A8CF","muted":"#8995BC","bold":"#D5D8EC","border_s":"#3a3f4f"}

if not runs:
    print("[INFO] No training runs found.")
    print("       Runs are discovered from trained_adapters_dir in settings")
    print("       and any additional history_output_roots.")
else:
    _rows = []
    for r in runs:
        name = r.get("run_name","?")[:34]
        adapter = r.get("adapter","?")[:7]
        ep = r.get("epochs", 0)
        best = r.get("best_loss")
        best_s = f"{best:.6f}" if best and isinstance(best,(int,float)) else "--"
        status = r.get("status","?")[:9]
        sc = _SS["success"] if status == "complete" else _SS["muted"]
        _rows.append(
            f'<tr style="border-bottom:1px solid {_SS["border_s"]};">'
            f'<td style="padding:4px 12px 4px 0;color:{_SS["bold"]};">{name}</td>'
            f'<td style="padding:4px 12px;color:{_SS["text"]};">{adapter}</td>'
            f'<td style="padding:4px 12px;color:{_SS["text"]};text-align:right;">{ep}</td>'
            f'<td style="padding:4px 12px;color:{_SS["text"]};text-align:right;">{best_s}</td>'
            f'<td style="padding:4px 12px;color:{sc};">{status}</td></tr>'
        )
    display(HTML(
        f'<div style="background:{_SS["panel"]};border:1px solid {_SS["border"]};'
        f'border-left:4px solid {_SS["primary"]};border-radius:6px;padding:16px 20px;margin:8px 0;">'
        f'<div style="color:{_SS["primary"]};font-weight:bold;margin-bottom:8px;">{len(runs)} Run(s)</div>'
        f'<table style="border-collapse:collapse;width:100%;">'
        f'<tr style="border-bottom:1px solid {_SS["border"]};">'
        f'<th style="text-align:left;padding:4px 12px 4px 0;color:{_SS["muted"]};">Run Name</th>'
        f'<th style="text-align:left;padding:4px 12px;color:{_SS["muted"]};">Adapter</th>'
        f'<th style="text-align:right;padding:4px 12px;color:{_SS["muted"]};">Epochs</th>'
        f'<th style="text-align:right;padding:4px 12px;color:{_SS["muted"]};">Best Loss</th>'
        f'<th style="text-align:left;padding:4px 12px;color:{_SS["muted"]};">Status</th></tr>'
        f'{"".join(_rows)}</table></div>'
    ))
